# 📊 Exploration Couche GOLD — Olist E-Commerce

Analyse des KPIs métier générés par la pipeline PySpark.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Configuration matplotlib pour des graphiques propres
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 2. Chargement des données GOLD

In [ ]:
# Chemins des fichiers Parquet
paths = {
    'monthly_revenue': '../data/gold/monthly_revenue/',
    'top_categories': '../data/gold/top_categories/',
    'top_sellers': '../data/gold/top_sellers/',
    'delay_impact': '../data/gold/delay_impact/',
    'payment_distribution': '../data/gold/payment_distribution/'
}

# Chargement des DataFrames
gold_data = {}
for name, path in paths.items():
    gold_data[name] = pd.read_parquet(path)
    print(f"✅ {name} chargé : {gold_data[name].shape[0]} lignes")

### Aperçu des données

In [ ]:
def show_data_summary(name, df):
    print(f"\n--- {name.upper()} ---")
    print("\nHead():")
    display(df.head())
    print("\nInfo():")
    df.info()
    print("\nValeurs nulles:")
    print(df.isnull().sum())

for name, df in gold_data.items():
    show_data_summary(name, df)

## 3. Data sanity check

In [ ]:
# Vérification rapide des doublons
for name, df in gold_data.items():
    duplicates = df.duplicated().sum()
    print(f"{name} : {duplicates} doublons")

## 4. Visualisations Business

### 📈 4.1 CA mensuel

In [ ]:
df_revenue = gold_data['monthly_revenue'].copy()

# Si la colonne n'est pas 'month' mais 'order_purchase_timestamp', on ajuste
if 'order_purchase_timestamp' in df_revenue.columns:
    df_revenue['month'] = pd.to_datetime(df_revenue['order_purchase_timestamp']).dt.to_period('M')
    df_revenue = df_revenue.groupby('month').agg({'revenue': 'sum'}).reset_index()
    df_revenue['month'] = df_revenue['month'].astype(str)

plt.figure(figsize=(14, 6))
plt.plot(df_revenue['month'], df_revenue['revenue'], marker='o', linewidth=2, color='#2ecc71')
plt.title('Évolution du chiffre d\'affaires mensuel', fontsize=14, pad=20)
plt.xlabel('Mois', fontsize=12)
plt.ylabel('Chiffre d\'affaires (R$)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 📊 4.2 Top 10 catégories produits

In [ ]:
df_categories = gold_data['top_categories'].head(10).copy()

plt.figure(figsize=(12, 6))
plt.barh(df_categories['product_category_name'], df_categories['revenue'], color='#3498db')
plt.gca().invert_yaxis()  # Pour avoir la catégorie la plus performante en haut
plt.title('Top 10 catégories par chiffre d\'affaires', fontsize=14, pad=20)
plt.xlabel('Chiffre d\'affaires (R$)', fontsize=12)
plt.ylabel('Catégorie produit', fontsize=12)
plt.tight_layout()
plt.show()

### 🏆 4.3 Top 10 vendeurs

In [ ]:
df_sellers = gold_data['top_sellers'].head(10).copy()

# On masque les IDs pour la confidentialité
df_sellers['seller_id'] = [f"Vendeur {i+1}" for i in range(len(df_sellers))]

plt.figure(figsize=(12, 6))
plt.barh(df_sellers['seller_id'], df_sellers['revenue'], color='#9b59b6')
plt.gca().invert_yaxis()
plt.title('Top 10 vendeurs par chiffre d\'affaires', fontsize=14, pad=20)
plt.xlabel('Chiffre d\'affaires (R$)', fontsize=12)
plt.ylabel('Vendeur', fontsize=12)
plt.tight_layout()
plt.show()

### 💳 4.4 Répartition des moyens de paiement

In [ ]:
df_payments = gold_data['payment_distribution'].copy()

plt.figure(figsize=(10, 10))
plt.pie(df_payments['total'], labels=df_payments['payment_type'], autopct='%1.1f%%', 
        startangle=90, colors=['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#95a5a6'])
plt.title('Répartition des moyens de paiement', fontsize=14, pad=20)
plt.show()

### ⏱ 4.5 Impact des retards sur la satisfaction client

In [ ]:
df_delay = gold_data['delay_impact'].copy()
df_delay['is_late'] = df_delay['is_late'].map({0: 'À l\'heure', 1: 'En retard'})

plt.figure(figsize=(10, 6))
plt.bar(df_delay['is_late'], df_delay['avg_review'], color=['#2ecc71', '#e74c3c'])
plt.title('Note moyenne selon le respect du délai de livraison', fontsize=14, pad=20)
plt.xlabel('Statut de livraison', fontsize=12)
plt.ylabel('Note moyenne (1-5)', fontsize=12)
plt.ylim(0, 5)
for i, v in enumerate(df_delay['avg_review']):
    plt.text(i, v + 0.1, f"{v:.2f}", ha='center', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Analyse métier

### 🔍 Insight 1 : Tendances CA

Le chiffre d'affaires semble avoir une tendance générale à la hausse, avec des pics saisonniers potentiels. Les mois de novembre et décembre (Black Friday/Cyber Monday) montrent souvent des pics significatifs.

### 🔍 Insight 2 : Catégories dominantes

Certaines catégories (comme beauté_sante, cama_mesa_banho) génèrent la majorité des revenus, indiquant des opportunités d'optimisation de stock et de marketing sur ces produits phares.

### 🔍 Insight 3 : Impact critique des retards

Les commandes livrées en retard ont une note moyenne bien inférieure (en général 2 points de moins). Cela démontre l'importance stratégique de la logistique pour la satisfaction et la fidélisation client.

### 🔍 Insight 4 : Comportement paiement

La carte de crédit est le moyen de paiement dominant (environ 80%), suivi du boleto bancário. Cela reflète les habitudes de consommation brésiliennes et justifie l'investissement dans des solutions de paiement adaptées.

## 6. Conclusion

### Résumé de la pipeline

Nous avons construit une pipeline PySpark complète en 4 couches :
- RAW : Fichiers CSV d'origine
- Bronze : Données ingestées en Parquet
- Silver : Données nettoyées et fiabilisées
- Gold : KPIs métier prêts à l'analyse

### KPIs clés

- Chiffre d'affaires total
- Chiffre d'affaires mensuel
- Panier moyen
- Top catégories et vendeurs
- Délai moyen et taux de retard
- Note moyenne et impact des retards
- Répartition des paiements

### Valeur métier

Le projet permet à Olist de :
- Comprendre les tendances de ventes
- Optimiser la logistique pour réduire les retards
- Identifier les produits et vendeurs performants
- Analyser le comportement client

### Limites du système

- Pipeline en mode local seulement
- Pas d'automatisation (Airflow)
- Pas de dashboard interactif
- Données statiques (pas de streaming)